# Cell Segmentation Training

Train the Lightning model and write a Kaggle submission from one notebook.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))
print(repo_root)

In [ ]:
# Run this once on Kaggle if the packages are not already available.
# Internet must be enabled for installation.
req_file = repo_root / "requirements-kaggle.txt"
%pip install -q -r {req_file}
%pip install -q -e {repo_root} --no-deps

In [ ]:
from hydra import compose, initialize_config_dir

with initialize_config_dir(version_base=None, config_dir=str(repo_root / "configs")):
    cfg = compose(
        config_name="train",
        overrides=[
            "paths.data_dir=/kaggle/input/tum-ai-e-lab-cell-segmentation/Archive",
            "paths.output_dir=/kaggle/working/outputs",
            "paths.submission_path=/kaggle/working/submission.csv",
            "data.batch_size=16",
            "data.num_workers=2",
            "trainer.max_epochs=25",
            "model.arch=FPN",
            "model.encoder_name=resnet34",
            # Use null if Kaggle internet is disabled and ImageNet weights are not cached.
            "model.encoder_weights=imagenet",
        ],
    )

cfg

In [ ]:
from cellseg_challenge.runner import train_from_config

best_ckpt = train_from_config(cfg)
best_ckpt

In [ ]:
from omegaconf import OmegaConf
from cellseg_challenge.runner import predict_from_config

pred_cfg = OmegaConf.create(OmegaConf.to_container(cfg, resolve=True))
pred_cfg.predict.ckpt_path = best_ckpt
submission_path = predict_from_config(pred_cfg)
submission_path